<table>
<tr>                                                                                   
     <th>
         <div style='padding:15px;color:#030aa7;font-size:240%;text-align: center;font-style: italic;font-weight: bold;font-family: Georgia, serif'>Spark-DataFrames SQL</div>
     </th>
     <th><img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/spark/presentation.png" width="256">
     </th>
 </tr>
</table>

In [1]:
from pyspark.sql import SparkSession
import os, warnings

warnings.filterwarnings(action="ignore")

In [2]:
spark = (SparkSession.builder
         .appName("09-Contrôler.la.cohérence.des.six.étoiles")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.legacy.charVarcharAsString", "true")     
         .config("spark.sql.legacy.parquet.nanosAsLong", "true")
         .getOrCreate())

print("Master :", spark.sparkContext.master)
print("Application :", spark.sparkContext.applicationId)
print("Python :", os.sys.version.split()[0])
print("Spark :", spark.version)

26/10/20 20:54:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/20 20:54:17 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/10/20 20:54:18 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to spark-events/eventlog_v2_app-20261020205416-0008/events_1_app-20261020205416-0008.zstd. This is Unsupported


Master : spark://spark-master:7077
Application : app-20261020205416-0008
Python : 3.10.12
Spark : 4.0.4


In [3]:
spark

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>Lecture et consolidation de données</div></b>

In [4]:
ROOT = "s3a://lakehouse"
donnees = {}
for nom in ["fact_logistics","fact_order","fact_order_item","fact_payment","fact_review","mart_customer_rfm"]:
    donnees[nom] = spark.read.parquet(f"{ROOT}/silver/ebrasil/{nom}")
    # print(donnees[nom].printSchema())

In [5]:
donnees["fact_logistics"].toPandas().head(5)

26/10/20 20:54:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,order_id,customer_id,customer_unique_id,representative_seller_id,customer_zip,seller_zip,status,purchase_date_key,approved_date_key,shipped_date_key,...,estimated_delivery_date_month,estimated_delivery_date_month_name,estimated_delivery_date_iso_week,estimated_delivery_date_day,estimated_delivery_date_weekday,estimated_delivery_date_is_weekend,purchase_time_hour,purchase_time_day_part,delivered_time_hour,delivered_time_day_part
0,41272756ecddd9a9ed0180413cc22fb6,914991f0c02ef0843c0e7010c819d642,0000f6ccb0745a6a4b88665a16c9f078,ef506c96320abeedfb894c34db06f478,66812,3569,delivered,20171012,20171012.0,20171013.0,...,11,November,46,13,2,False,20,evening,21.0,evening
1,d957021f1127559cd947b62533f484f7,47227568b10f5f58a524a75507e6992c,0004aac84e0df4da2b147fca70cf8255,70a12e78e608ac31179aea7f8422044b,18040,12327,delivered,20171114,20171114.0,20171116.0,...,12,December,49,5,3,False,19,evening,23.0,evening
2,d0028facea13f508e880202d7097a5a1,d2509c13692836fc0449e88cf9eb4858,00050ab1314c0e55a6ca13cf7181fecf,ea8482cd71df3c1969d7b9473ff13abc,13084,4160,delivered,20180420,20180425.0,20180425.0,...,5,May,19,9,4,False,12,afternoon,12.0,afternoon
3,ae76bef74b97bcb0b3e355e60d9a6f9c,3b37fb626fdf46cd99d37ec62afa88ff,0005e1862207bf6ccc02e4228effd9a0,0cbcee27c791afa0cdcb08587a2013a8,25966,37410,delivered,20170304,20170304.0,20170306.0,...,4,April,14,6,5,False,23,evening,8.0,morning
4,6681163e3dab91c549952b2845b20281,2f29573c8cac5a7be11c5b649078f944,0006fdc98a402fceb4eb0ee528f6a8d4,d2374cbcbb3ca4ab1086534108cc3ab7,29400,14940,delivered,20170718,20170718.0,20170721.0,...,8,August,33,15,3,False,9,morning,18.0,evening


In [6]:
donnees["fact_order"].toPandas().head(5)

,order_id,customer_unique_id,status,purchase_date_key,approved_date_key,shipped_date_key,delivered_date_key,estimated_delivery_date_key,purchase_time_key,approved_time_key,...,estimated_delivery_date_weekday,estimated_delivery_date_is_weekend,purchase_time_hour,purchase_time_day_part,approved_time_hour,approved_time_day_part,shipped_time_hour,shipped_time_day_part,delivered_time_hour,delivered_time_day_part
0,41272756ecddd9a9ed0180413cc22fb6,0000f6ccb0745a6a4b88665a16c9f078,delivered,20171012,20171012.0,20171013.0,20171101.0,20171113,200000,200000.0,...,2,False,20,evening,20.0,evening,20.0,evening,21.0,evening
1,d957021f1127559cd947b62533f484f7,0004aac84e0df4da2b147fca70cf8255,delivered,20171114,20171114.0,20171116.0,20171127.0,20171205,190000,200000.0,...,3,False,19,evening,20.0,evening,19.0,evening,23.0,evening
2,d0028facea13f508e880202d7097a5a1,00050ab1314c0e55a6ca13cf7181fecf,delivered,20180420,20180425.0,20180425.0,20180427.0,20180509,120000,30000.0,...,4,False,12,afternoon,3.0,night,15.0,afternoon,12.0,afternoon
3,ae76bef74b97bcb0b3e355e60d9a6f9c,0005e1862207bf6ccc02e4228effd9a0,delivered,20170304,20170304.0,20170306.0,20170309.0,20170406,230000,230000.0,...,5,False,23,evening,23.0,evening,5.0,night,8.0,morning
4,6681163e3dab91c549952b2845b20281,0006fdc98a402fceb4eb0ee528f6a8d4,delivered,20170718,20170718.0,20170721.0,20170803.0,20170815,90000,90000.0,...,3,False,9,morning,9.0,morning,16.0,afternoon,18.0,evening


In [7]:
donnees["fact_order_item"].toPandas().head(5)

,order_id,order_item_id,purchase_timestamp,purchase_date_key,purchase_time_key,customer_unique_id,product_id,category_name,seller_id,price,...,purchase_date_semester,purchase_date_quarter,purchase_date_month,purchase_date_month_name,purchase_date_iso_week,purchase_date_day,purchase_date_weekday,purchase_date_is_weekend,purchase_time_hour,purchase_time_day_part
0,41272756ecddd9a9ed0180413cc22fb6,1,2017-10-12 20:00:00,20171012,200000,0000f6ccb0745a6a4b88665a16c9f078,2345a354a6f2033609bbf62bf5be9ef6,telephony,ef506c96320abeedfb894c34db06f478,25.99,...,2,4,10,October,41,12,5,False,20,evening
1,d957021f1127559cd947b62533f484f7,1,2017-11-14 19:00:00,20171114,190000,0004aac84e0df4da2b147fca70cf8255,c72e18b3fe2739b8d24ebf3102450f37,telephony,70a12e78e608ac31179aea7f8422044b,180.00,...,2,4,11,November,46,14,3,False,19,evening
2,d0028facea13f508e880202d7097a5a1,1,2018-04-20 12:00:00,20180420,120000,00050ab1314c0e55a6ca13cf7181fecf,8cefe1c6f2304e7e6825150218ffc58c,telephony,ea8482cd71df3c1969d7b9473ff13abc,27.99,...,1,2,4,April,16,20,6,False,12,afternoon
3,ae76bef74b97bcb0b3e355e60d9a6f9c,1,2017-03-04 23:00:00,20170304,230000,0005e1862207bf6ccc02e4228effd9a0,e24f73b7631ee3fbb2ab700a9acaa258,fashion_bags_accessories,0cbcee27c791afa0cdcb08587a2013a8,135.00,...,1,1,3,March,9,4,7,True,23,evening
4,6681163e3dab91c549952b2845b20281,1,2017-07-18 09:00:00,20170718,90000,0006fdc98a402fceb4eb0ee528f6a8d4,7ca5f56188492e30c50ba64de188aae7,bed_bath_table,d2374cbcbb3ca4ab1086534108cc3ab7,13.90,...,2,3,7,July,29,18,3,False,9,morning


In [8]:
donnees["fact_payment"].toPandas().head(5)

,order_id,payment_sequential,payment_type,customer_unique_id,purchase_date_key,purchase_time_key,payment_value,installments,transaction_count,event_ts,...,purchase_date_semester,purchase_date_quarter,purchase_date_month,purchase_date_month_name,purchase_date_iso_week,purchase_date_day,purchase_date_weekday,purchase_date_is_weekend,purchase_time_hour,purchase_time_day_part
0,41272756ecddd9a9ed0180413cc22fb6,1,credit_card,0000f6ccb0745a6a4b88665a16c9f078,20171012,200000,43.62,4,1,2017-10-12T20:00:00Z,...,2,4,10,October,41,12,5,False,20,evening
1,d957021f1127559cd947b62533f484f7,1,credit_card,0004aac84e0df4da2b147fca70cf8255,20171114,190000,196.89,6,1,2017-11-14T19:00:00Z,...,2,4,11,November,46,14,3,False,19,evening
2,d0028facea13f508e880202d7097a5a1,1,boleto,00050ab1314c0e55a6ca13cf7181fecf,20180420,120000,35.38,1,1,2018-04-20T12:00:00Z,...,1,2,4,April,16,20,6,False,12,afternoon
3,ae76bef74b97bcb0b3e355e60d9a6f9c,1,credit_card,0005e1862207bf6ccc02e4228effd9a0,20170304,230000,150.12,3,1,2017-03-04T23:00:00Z,...,1,1,3,March,9,4,7,True,23,evening
4,6681163e3dab91c549952b2845b20281,1,credit_card,0006fdc98a402fceb4eb0ee528f6a8d4,20170718,90000,29.00,2,1,2017-07-18T09:00:00Z,...,2,3,7,July,29,18,3,False,9,morning


In [9]:
donnees["fact_review"].toPandas().head(5)

,order_id,customer_unique_id,status,review_count,score_sum,average_score,text_review_groups,answer_hours_sum,creation_hours_sum,purchase_date_key,...,purchase_date_semester,purchase_date_quarter,purchase_date_month,purchase_date_month_name,purchase_date_iso_week,purchase_date_day,purchase_date_weekday,purchase_date_is_weekend,purchase_time_hour,purchase_time_day_part
0,41272756ecddd9a9ed0180413cc22fb6,0000f6ccb0745a6a4b88665a16c9f078,delivered,1,4,4.0,1,24.0,483.0,20171012,...,2,4,10,October,41,12,5,False,20,evening
1,d957021f1127559cd947b62533f484f7,0004aac84e0df4da2b147fca70cf8255,delivered,1,5,5.0,1,118.0,316.0,20171114,...,2,4,11,November,46,14,3,False,19,evening
2,d0028facea13f508e880202d7097a5a1,00050ab1314c0e55a6ca13cf7181fecf,delivered,1,4,4.0,1,92.0,179.0,20180420,...,1,2,4,April,16,20,6,False,12,afternoon
3,ae76bef74b97bcb0b3e355e60d9a6f9c,0005e1862207bf6ccc02e4228effd9a0,delivered,1,4,4.0,1,147.0,120.0,20170304,...,1,1,3,March,9,4,7,True,23,evening
4,6681163e3dab91c549952b2845b20281,0006fdc98a402fceb4eb0ee528f6a8d4,delivered,1,3,3.0,1,119.0,398.0,20170718,...,2,3,7,July,29,18,3,False,9,morning


In [10]:
donnees["mart_customer_rfm"].toPandas().head(5)

,customer_unique_id,last_purchase,frequency,monetary,average_order_value,recency_days,last_purchase_date_key,last_purchase_time_key,event_ts,customer_customer_id,...,last_purchase_date_semester,last_purchase_date_quarter,last_purchase_date_month,last_purchase_date_month_name,last_purchase_date_iso_week,last_purchase_date_day,last_purchase_date_weekday,last_purchase_date_is_weekend,last_purchase_time_hour,last_purchase_time_day_part
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:00:00,1,141.90,141.90,3085,20180510,100000,2018-05-10T10:00:00Z,fadbb3709178fc513abc1b2670aa1ad2,...,1,2,5,May,19,10,5,False,10,morning
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:00:00,1,27.19,27.19,3088,20180507,110000,2018-05-07T11:00:00Z,4cb282e167ae9234755102258dd52ee8,...,1,2,5,May,19,7,2,False,11,morning
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:00:00,1,86.22,86.22,3511,20170310,210000,2017-03-10T21:00:00Z,9b3932a6253894a02c1df9d19004239f,...,1,1,3,March,10,10,6,False,21,evening
3,000bfa1d2f1a41876493be685390d6d3,2017-09-29 14:00:00,1,46.85,46.85,3308,20170929,140000,2017-09-29T14:00:00Z,e36917a47d5bf39bd3f38bcafb3d438f,...,2,3,9,September,39,29,6,False,14,afternoon
4,001147e649a7b1afd577e873841632dd,2017-08-31 17:00:00,1,212.16,212.16,3337,20170831,170000,2017-08-31T17:00:00Z,5247e5c7e9037e74343f13bbd8800a6a,...,2,3,8,August,35,31,5,False,17,afternoon


# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>fact_order</div></b>

<table>
<tr>                                                                                   
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/postgreSQL/ebrasil_pg.png" width="1024">
     </th>
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/postgreSQL/fact_order.png" width="1024">
     </th>
 </tr>
</table>


Le grain prévu est une commande (order_id) étoile valide mais une jointure peut dupliquer des lignes

In [11]:
print(f"enregistrements\t{donnees['fact_order'].select('order_id').count()}\ncommandes\t{donnees['fact_order'].select('order_id').distinct().count()} ")

enregistrements	99441
commandes	99441 


# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>fact_order_item</div></b>
<table>
<tr>                                                                                   
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/postgreSQL/ebrasil_pg.png" width="1024">
     </th>
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/postgreSQL/fact_order_item.png" width="1024">
     </th>
 </tr>
</table>

Le grain prévu est une ligne par article de commande (order_id,order_item_id) étoile valide mais une jointure peut dupliquer des lignes

In [12]:
print(f"enregistrements      \t{donnees['fact_order_item'].select(['order_id','order_item_id']).count()}\narticles par commandes\t{donnees['fact_order_item'].select(['order_id','order_item_id']).distinct().count()} ")

enregistrements      	112650
articles par commandes	112650 


In [13]:
donnees['fact_order_item'].count(), donnees['fact_order_item'].select(['order_id']).distinct().count()

(112650, 98666)

In [14]:
donnees['items'] = spark.read.parquet("s3a://lakehouse/bronze/ebrasil/items.parquet")

In [15]:
donnees['items'].select(['order_id']).count(), donnees['items'].select(['order_id']).distinct().count()

(112650, 98666)

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>fact_payment</div></b>
<table>
<tr>                                                                                   
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/postgreSQL/ebrasil_pg.png" width="1024">
     </th>
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/postgreSQL/fact_payment.png" width="1024">
     </th>
 </tr>
</table>

Le grain prévu est une ligne par article de commande (order_id,order_item_id) étoile valide mais une jointure peut dupliquer des lignes

In [16]:
donnees['fact_payment'].count(), donnees['fact_payment'].select('order_id').count(), donnees['fact_payment'].select('order_id').distinct().count(),donnees['fact_payment'].select('order_id').distinct().count()  

(103886, 103886, 99440, 99440)

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>fact_review</div></b>
<table>
<tr>                                                                                   
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/postgreSQL/ebrasil_pg.png" width="1024">
     </th>
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/fix/ebrasil-fact-payment-grain/images/postgreSQL/fact_review.png" width="1024">
     </th>
 </tr>
</table>

Le grain prévu est une ligne par article de commande (order_id,order_item_id) étoile valide mais une jointure peut dupliquer des lignes

In [17]:
donnees['fact_review'].count(), donnees['fact_review'].select('order_id').count(), donnees['fact_review'].select('order_id').distinct().count(),donnees['fact_review'].select('order_id').distinct().count()  

(100000, 100000, 99441, 99441)

# <b><div style='padding:15px;background-color:#d8dcd6;color:#030aa7;font-size:120%;text-align: left'>fact_logistics</div></b>
<table>
<tr>                                                                                   
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/master/images/postgreSQL/ebrasil_pg.png" width="1024">
     </th>
     <th>
         <img src="https://raw.githubusercontent.com/rbizoi/FormationBigDataBI/refs/heads/fix/ebrasil-fact-payment-grain/images/postgreSQL/fact_logistics.png" width="1024">
     </th>
 </tr>
</table>

Le grain prévu est une ligne par article de commande (order_id,order_item_id) étoile valide mais une jointure peut dupliquer des lignes

In [18]:
donnees['fact_logistics'].count(), donnees['fact_logistics'].select('order_id').count(), donnees['fact_logistics'].select('order_id').distinct().count(),donnees['fact_logistics'].select('order_id').distinct().count()  

(99441, 99441, 99441, 99441)